# Experiment: Roadrunner EGP Phase-60 Gravity Sweep

Goal:

- Fix the planet/star condition at `T=500 K`, `a=5 AU`, `Rp=1.0 Rj`, and `phase=60 deg`.
- Sweep only the EGP gravity code: `g31`, `g100`, `g126`, `g316`, `g1000`.
- Use matching EGP/SLGRID PT + cloud files for reflected light and matching EGP IRflux files for thermal flux.
- Produce the main table and figure: reflected fraction vs gravity code for CGI-1, CGI-2, CGI-3, and CGI-4.


In [ ]:
# Setup: PICASO4-local imports and paths
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
SEARCH_ROOTS.append(Path("/Users/xin/Documents/Documents/College/aurora"))

for candidate in SEARCH_ROOTS:
    if (candidate / "roadrunner_egp" / "roadrunner").exists():
        REPO_ROOT = candidate.resolve()
        break
else:
    raise FileNotFoundError("Could not locate aurora/roadrunner_egp/roadrunner.")

ROADRUNNER_ROOT = REPO_ROOT / "roadrunner_egp"
SCIENCE_INPUTS = REPO_ROOT / "science_inputs"
RESULTS_DIR = ROADRUNNER_ROOT / "results" / "phase60_t500_a5_gravity_sweep"
TMP_DIR = REPO_ROOT / "tmp"

for path in (RESULTS_DIR, TMP_DIR / "matplotlib", TMP_DIR / "numba-cache"):
    path.mkdir(parents=True, exist_ok=True)

# Force this notebook to use the Aurora data.
os.environ["ROADRUNNER_SCIENCE_INPUTS"] = str(SCIENCE_INPUTS)
os.environ["picaso_refdata"] = str(REPO_ROOT / "picaso4_reference")
os.environ["PYSYN_CDBS"] = str(REPO_ROOT / "picaso4_reference" / "stellar_grids")
os.environ["SLGRID_BASE_DIR"] = str(SCIENCE_INPUTS / "slgrid")
os.environ["SLGRID_PT_DIR"] = str(SCIENCE_INPUTS / "slgrid" / "climate")
os.environ["SLGRID_CLD_DIR"] = str(SCIENCE_INPUTS / "slgrid" / "clouds")
os.environ["EGP_IRFLUX_DIR"] = str(SCIENCE_INPUTS / "egp" / "irflux")
os.environ["MPLCONFIGDIR"] = str(TMP_DIR / "matplotlib")
os.environ["NUMBA_CACHE_DIR"] = str(TMP_DIR / "numba-cache")

if str(ROADRUNNER_ROOT) not in sys.path:
    sys.path.insert(0, str(ROADRUNNER_ROOT))

# Make repeated top-to-bottom notebook runs use the env vars above.
for module_name in list(sys.modules):
    if (
        module_name == "roadrunner"
        or module_name.startswith("roadrunner.")
        or module_name == "workflows.phase60_gravity_sweep"
        or module_name == "workflows.hybrid_reflected_picaso_thermal_egp"
    ):
        del sys.modules[module_name]

from workflows.phase60_gravity_sweep import (
    HAVE_PICASO,
    REFLECT_THRESHOLD,
    gravity_code_to_logg_cgs,
    gravity_sweep_pivot,
    phase60_gravity_file_inventory,
    plot_phase60_gravity_sweep,
    run_phase60_gravity_sweep,
)
from roadrunner.config import (
    CGI_BANDS,
    EGP_IRFLUX_DIR,
    LAM_GRID,
    SLGRID_CLD_DIR,
    SLGRID_PT_DIR,
)

print(f"Repo root: {REPO_ROOT}")
print(f"Notebook results: {RESULTS_DIR}")
print(f"PICASO available: {HAVE_PICASO}")
print(f"picaso_refdata: {os.environ['picaso_refdata']}")
print(f"PYSYN_CDBS: {os.environ['PYSYN_CDBS']}")
print(f"SLGRID climate: {SLGRID_PT_DIR}")
print(f"SLGRID clouds: {SLGRID_CLD_DIR}")
print(f"EGP IRflux: {EGP_IRFLUX_DIR}")
print(f"Wavelength grid: {LAM_GRID.min():.2f}-{LAM_GRID.max():.2f} um ({len(LAM_GRID)} pts)")
print(f"CGI bands: {list(CGI_BANDS.keys())}")


## Fixed Run Configuration

This is the phase-60 version of the old phase-angle sweep flow. The swept variable is now `gravity_code`; all other physical settings are fixed.


In [ ]:
TEMPERATURE_K = 500
GRAVITY_CODES = ["31", "100", "126", "316", "1000"]
METALLICITY = "+000"
CO_RATIO = "100"
FSED = "3"
FRAC = None

PHASE_DEG = 60.0
R_PLANET_RJ = 1.0
SEMI_MAJOR_AU = 5.0

OUTPUT_CSV = RESULTS_DIR / "roadrunner_egp_phase60_T500_a5_gravity_sweep.csv"
OUTPUT_PNG = RESULTS_DIR / "roadrunner_egp_phase60_T500_a5_gravity_sweep.png"

config_summary = pd.DataFrame(
    [
        {"parameter": "temperature_k", "value": TEMPERATURE_K},
        {"parameter": "gravity_codes", "value": GRAVITY_CODES},
        {"parameter": "logg_cgs", "value": [round(gravity_code_to_logg_cgs(code), 6) for code in GRAVITY_CODES]},
        {"parameter": "metallicity", "value": METALLICITY},
        {"parameter": "co_ratio", "value": CO_RATIO},
        {"parameter": "fsed", "value": FSED},
        {"parameter": "frac", "value": FRAC},
        {"parameter": "phase_deg", "value": PHASE_DEG},
        {"parameter": "r_planet_rj", "value": R_PLANET_RJ},
        {"parameter": "semi_major_au", "value": SEMI_MAJOR_AU},
        {"parameter": "thermal_source", "value": "egp_irflux"},
        {"parameter": "atmosphere_source", "value": "slgrid"},
        {"parameter": "threshold", "value": REFLECT_THRESHOLD},
    ]
)

display(config_summary)


## Preflight: Exact EGP File Inventory

The run should fail here if any gravity code is missing its exact `T500`, `m+000`, `CO100`, `fsed3` PT, cloud, or IRflux file.


In [ ]:
file_inventory = phase60_gravity_file_inventory(
    temperature_k=TEMPERATURE_K,
    gravity_codes=GRAVITY_CODES,
    metallicity=METALLICITY,
    co_ratio=CO_RATIO,
    fsed=FSED,
    frac=FRAC,
)

display(file_inventory)

missing_mask = ~(
    file_inventory["pt_exists"]
    & file_inventory["cld_exists"]
    & file_inventory["egp_irflux_exists"]
)
if missing_mask.any():
    raise FileNotFoundError("Missing at least one exact EGP input file. See file_inventory above.")

print("All exact T500 EGP PT/cloud/IRflux files are present.")


## Main Run

This cell runs one fixed phase-60 case per gravity code and saves a combined CSV.


In [ ]:
if not HAVE_PICASO:
    raise RuntimeError("PICASO is not available in this notebook kernel.")

sweep_df = run_phase60_gravity_sweep(
    teff_k=TEMPERATURE_K,
    rj=R_PLANET_RJ,
    a_au=SEMI_MAJOR_AU,
    gravity_codes=GRAVITY_CODES,
    phase_deg=PHASE_DEG,
    metallicity=METALLICITY,
    co_ratio=CO_RATIO,
    fsed=FSED,
    frac=FRAC,
)

sweep_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved detailed gravity-sweep results to: {OUTPUT_CSV}")
print(f"Rows: {len(sweep_df)}")

display(sweep_df)


## Pivot Table And Main Figure


In [ ]:
gravity_table = gravity_sweep_pivot(sweep_df)
display(gravity_table)

ax = plot_phase60_gravity_sweep(sweep_df)
plt.tight_layout()
ax.figure.savefig(OUTPUT_PNG, dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved gravity-sweep plot to: {OUTPUT_PNG}")


## Validation


In [ ]:
expected_rows = len(GRAVITY_CODES) * len(CGI_BANDS)

assert len(sweep_df) == expected_rows, f"Expected {expected_rows} rows, got {len(sweep_df)}"
assert sorted(sweep_df["gravity_code"].astype(str).unique()) == sorted(GRAVITY_CODES)
assert sweep_df["phase_deg"].nunique() == 1
assert float(sweep_df["phase_deg"].iloc[0]) == PHASE_DEG
assert sweep_df["T_eff"].nunique() == 1
assert float(sweep_df["T_eff"].iloc[0]) == float(TEMPERATURE_K)
assert sweep_df["a_AU"].nunique() == 1
assert float(sweep_df["a_AU"].iloc[0]) == float(SEMI_MAJOR_AU)
assert sweep_df["f_reflect"].notna().all()
assert OUTPUT_CSV.exists()
assert OUTPUT_PNG.exists()

expected_irflux = {
    f"SLGRID_T{TEMPERATURE_K}_g{code}_m{METALLICITY}_CO{CO_RATIO}_fsed{FSED}_IRflux.txt"
    for code in GRAVITY_CODES
}
assert set(sweep_df["egp_irflux_file"]) == expected_irflux

print("Validation passed for fixed phase-60 EGP gravity sweep.")
print(f"CSV: {OUTPUT_CSV}")
print(f"PNG: {OUTPUT_PNG}")
